# 02 Embeddings And Pairs
Embeddings erzeugen/laden, QC, LSPO/ADS Pair-Building.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = "replace_with_run_id_from_00"
USE_STUB_EMBEDDINGS = False
PREFER_PRECOMPUTED_ADS = True
DEVICE = "auto"


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
import json
import numpy as np
import pandas as pd

from src.common.config import load_notebook_context, build_run_dirs
from src.common.io_schema import read_parquet, save_parquet
from src.features.embed_chars2vec import get_or_create_chars2vec_embeddings
from src.features.embed_specter import get_or_create_specter_embeddings
from src.approaches.nand.build_pairs import assign_lspo_splits, build_pairs_within_blocks, write_pairs

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]

RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")


In [3]:
rep = MODEL["representation"]

lspo_chars = get_or_create_chars2vec_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
lspo_text = get_or_create_specter_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=16,
    device=DEVICE,
    prefer_precomputed=False,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

ads_chars = get_or_create_chars2vec_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
ads_text = get_or_create_specter_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=32,
    device=DEVICE,
    prefer_precomputed=PREFER_PRECOMPUTED_ADS,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

print("lspo chars/text:", lspo_chars.shape, lspo_text.shape)
print("ads chars/text:", ads_chars.shape, ads_text.shape)


tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rapha\.cache\huggingface\hub\models--allenai--specter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

lspo chars/text: (1000, 50) (1000, 768)
ads chars/text: (1000, 50) (1000, 768)


In [4]:
def emb_qc(name, arr):
    norms = np.linalg.norm(arr, axis=1)
    return {
        "name": name,
        "shape": tuple(arr.shape),
        "nan_count": int(np.isnan(arr).sum()),
        "norm_mean": float(np.mean(norms)),
        "norm_p01": float(np.quantile(norms, 0.01)),
        "norm_p99": float(np.quantile(norms, 0.99)),
    }

pd.DataFrame([
    emb_qc("lspo_chars", lspo_chars),
    emb_qc("lspo_text", lspo_text),
    emb_qc("ads_chars", ads_chars),
    emb_qc("ads_text", ads_text),
])


,name,shape,nan_count,norm_mean,norm_p01,norm_p99
0,lspo_chars,"(1000, 50)",0,3.672508,2.798476,4.424648
1,lspo_text,"(1000, 768)",0,22.548742,21.580319,23.324893
2,ads_chars,"(1000, 50)",0,3.308096,2.423481,4.116272
3,ads_text,"(1000, 768)",0,22.340984,20.773095,23.280642


In [5]:
lspo_mentions = assign_lspo_splits(lspo_mentions, seed=int(RUN.get("seed", 11)))

lspo_pairs = build_pairs_within_blocks(
    mentions=lspo_mentions,
    max_pairs_per_block=RUN.get("max_pairs_per_block"),
    seed=int(RUN.get("seed", 11)),
    require_same_split=True,
    labeled_only=False,
    balance_train=True,
)

lspo_pairs_path = subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet"
write_pairs(lspo_pairs, lspo_pairs_path)
save_parquet(lspo_mentions, subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet", index=False)

print("LSPO pairs:", len(lspo_pairs), "->", lspo_pairs_path)


LSPO pairs: 457 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\replace_with_run_id_from_00\lspo_pairs_smoke.parquet


In [6]:
ads_pairs = build_pairs_within_blocks(
    mentions=ads_mentions,
    max_pairs_per_block=RUN.get("max_pairs_per_block"),
    seed=int(RUN.get("seed", 11)),
    require_same_split=False,
    labeled_only=False,
    balance_train=False,
)
ads_pairs_path = subset_dir / f"ads_pairs_{RUN_STAGE}.parquet"
write_pairs(ads_pairs, ads_pairs_path)
print("ADS pairs:", len(ads_pairs), "->", ads_pairs_path)


ADS pairs: 500 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\replace_with_run_id_from_00\ads_pairs_smoke.parquet


In [7]:
q = lspo_pairs.copy()
if "label" in q.columns:
    known = q[q["label"].notna()]
    pos = int((known["label"] == 1).sum())
    neg = int((known["label"] == 0).sum())
    ratio = pos / max(1, neg)
    print({"labeled_pairs": len(known), "pos": pos, "neg": neg, "pos_neg_ratio": ratio})

print("Pairs per block:")
print(q.groupby("block_key").size().describe())

leak = 0
if "orcid" in lspo_mentions.columns and "split" in lspo_mentions.columns:
    g = lspo_mentions[lspo_mentions["orcid"].notna()].groupby("orcid")["split"].nunique()
    leak = int((g > 1).sum())
print("orcid leakage groups:", leak)

with (metrics_dir / "02_pairs_qc.json").open("w", encoding="utf-8") as f:
    json.dump({"orcid_leakage_groups": leak, "lspo_pairs": int(len(lspo_pairs)), "ads_pairs": int(len(ads_pairs))}, f, indent=2)


{'labeled_pairs': 457, 'pos': 378, 'neg': 79, 'pos_neg_ratio': 4.784810126582278}
Pairs per block:
count    457.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
dtype: float64
orcid leakage groups: 0
